# United States Top 50 Playlist Analytics — Feature Engineering

## Objective

This notebook transforms the validated playlist dataset into analytical features used to measure song longevity, ranking performance, ranking stability, popularity, and song characteristics.

The engineered features will support song-level, artist-level, and popularity analysis in later stages of the project.

In [1]:
import pandas as pd
import numpy as np

## 1. Load Cleaned Dataset

In [2]:
DATA_PATH = "../data/processed/us_top50_clean.csv"

df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully!
Rows: 27700
Columns: 10


## 2. Song Duration Feature

The original dataset stores song duration in milliseconds. We convert it into minutes for easier interpretation and visualization.

In [4]:
df["duration_minutes"] = df["duration_ms"] / 60000

df[[
    "song",
    "duration_ms",
    "duration_minutes"
]].head()

,song,duration_ms,duration_minutes
0,Ella Baila Sola,165671,2.761183
1,Last Night,163854,2.730900
2,All My Life (feat. J. Cole),223878,3.731300
3,un x100to,194563,3.242717
4,Kill Bill,153946,2.565767


## 3. Song-Level Performance Metrics

The dataset contains multiple daily observations for each song. We aggregate these observations to calculate overall song performance across the available playlist history.

In [5]:
song_performance = (
    df
    .groupby(["song", "artist"])
    .agg(
        days_on_chart=("date", "nunique"),
        average_rank=("position", "mean"),
        best_rank=("position", "min"),
        worst_rank=("position", "max"),
        rank_volatility=("position", "std"),
        average_popularity=("popularity", "mean"),
        popularity_volatility=("popularity", "std"),
        average_duration_minutes=("duration_minutes", "mean")
    )
    .reset_index()
)

print("Unique song-artist combinations:",
      len(song_performance))

song_performance.head()

Unique song-artist combinations: 977


,song,artist,days_on_chart,average_rank,best_rank,worst_rank,rank_volatility,average_popularity,popularity_volatility,average_duration_minutes
0,"""Slut!"" (Taylor's Version) (From The Vault)",Taylor Swift,26,15.500000,3,38,12.130128,86.769231,17.781581,3.006350
1,(Don't Fear) The Reaper,Blue Öyster Cult,1,50.000000,50,50,NaN,77.000000,NaN,5.135333
2,(There's No Place Like) Home for the Holidays ...,Perry Como,2,45.000000,45,45,0.000000,70.000000,0.000000,2.931550
3,16 CARRIAGES,Beyoncé,10,26.000000,16,49,10.635371,64.500000,12.002315,3.829307
4,2 hands,Tate McRae,7,33.571429,8,45,12.286268,76.571429,3.207135,3.030333


In [6]:
song_performance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 977 entries, 0 to 976
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   song                      977 non-null    object 
 1   artist                    977 non-null    object 
 2   days_on_chart             977 non-null    int64  
 3   average_rank              977 non-null    float64
 4   best_rank                 977 non-null    int64  
 5   worst_rank                977 non-null    int64  
 6   rank_volatility           820 non-null    float64
 7   average_popularity        977 non-null    float64
 8   popularity_volatility     820 non-null    float64
 9   average_duration_minutes  977 non-null    float64
dtypes: float64(5), int64(3), object(2)
memory usage: 76.5+ KB


In [7]:
song_performance.describe()

,days_on_chart,average_rank,best_rank,worst_rank,rank_volatility,average_popularity,popularity_volatility,average_duration_minutes
count,977.000000,977.000000,977.000000,977.000000,820.000000,977.000000,820.000000,977.000000
mean,28.352098,30.050256,20.326510,42.026612,9.181462,75.020895,9.542196,3.428240
std,58.265423,10.744455,14.241364,10.235111,4.860504,17.601284,12.897261,0.990499
min,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.621900
25%,3.000000,22.613035,8.000000,39.000000,5.957200,69.000000,1.905577,2.785183
50%,7.000000,29.954545,18.000000,46.000000,9.434626,79.428571,3.332911,3.366683
75%,27.000000,38.500000,32.000000,49.000000,12.382707,86.000000,11.224384,3.909767
max,535.000000,50.000000,50.000000,50.000000,29.698485,99.202532,57.982756,12.333500


## 4. Handle Single-Observation Volatility

Some songs appear only once in the available playlist history. Their standard deviation is undefined because there is only one observation.

For analytical purposes, these missing volatility values are set to 0, representing no observed movement.

In [9]:
song_performance["rank_volatility"] = (
    song_performance["rank_volatility"].fillna(0)
)

song_performance["popularity_volatility"] = (
    song_performance["popularity_volatility"].fillna(0)
)

print("Missing rank volatility:",
    song_performance["rank_volatility"].isna().sum())

print("Missing popularity volatility:",
    song_performance["popularity_volatility"].isna().sum())

Missing rank volatility: 0
Missing popularity volatility: 0


## 5. Finalize Song-Level Metrics

Numerical performance metrics are rounded to improve readability in tables and dashboard outputs.

In [10]:
song_performance["average_rank"] = song_performance["average_rank"].round(2)
song_performance["rank_volatility"] = song_performance["rank_volatility"].round(2)
song_performance["average_popularity"] = song_performance["average_popularity"].round(2)
song_performance["popularity_volatility"] = song_performance["popularity_volatility"].round(2)
song_performance["average_duration_minutes"] = song_performance["average_duration_minutes"].round(2)

## 6. Artist-Level Performance Metrics

Artist-level metrics summarize how frequently artists appear on the Top 50 playlist and how many unique songs they contribute.

In [11]:
artist_performance = (
    df
    .groupby("artist")
    .agg(
        unique_songs=("song", "nunique"),
        total_chart_days=("date", "nunique"),
        average_rank=("position", "mean"),
        best_rank=("position", "min"),
        average_popularity=("popularity", "mean")
    )
    .reset_index()
)

artist_performance["average_rank"] = (
    artist_performance["average_rank"].round(2)
)

artist_performance["average_popularity"] = (
    artist_performance["average_popularity"].round(2)
)

print("Unique artists:", len(artist_performance))

artist_performance.head(10)

Unique artists: 297


,artist,unique_songs,total_chart_days,average_rank,best_rank,average_popularity
0,$uicideboy$,1,5,40.60,24,57.00
1,*NSYNC,1,16,36.31,24,85.25
2,21 Savage,13,115,20.24,2,89.08
3,4batz,1,52,37.10,25,85.10
4,4batz & Drake,1,13,33.23,16,82.31
5,Addison Rae,1,29,38.86,26,87.38
6,Alan Jackson,1,2,24.00,12,66.00
7,Andrew Gold,2,2,24.67,11,75.33
8,André 3000,1,1,31.00,31,37.00
9,Andy Williams,2,42,15.07,4,76.24


## 7. Artist Dominance Index

The Artist Dominance Index measures an artist's share of total playlist appearances across the available historical dataset.

A higher value indicates that the artist accounts for a larger share of playlist observations.

In [16]:
artist_appearances = (
    df.groupby("artist")
    .size()
    .reset_index(name="playlist_appearances")
)

artist_performance = artist_performance.merge(
    artist_appearances,
    on="artist",
    how="left"
)

total_playlist_observations = len(df)

artist_performance["artist_dominance_index"] = (
    artist_performance["playlist_appearances"]
    / total_playlist_observations
).round(4)

artist_performance.sort_values(
    "artist_dominance_index",
    ascending=False
).head(10)

,artist,unique_songs,total_chart_days,average_rank,best_rank,average_popularity,artist_dominance_index,playlist_appearances
242,Taylor Swift,96,437,24.56,1,87.10,0.0727,2014
286,Zach Bryan,35,545,25.62,1,86.02,0.0669,1854
177,Morgan Wallen,10,539,30.61,1,85.41,0.0601,1666
229,Sabrina Carpenter,14,315,17.60,1,90.20,0.0349,966
65,Drake,35,324,26.26,1,84.61,0.0303,838
40,Chappell Roan,6,232,20.65,1,88.66,0.0287,794
266,Travis Scott,31,331,25.74,1,88.59,0.0274,760
22,Billie Eilish,12,333,20.36,2,92.77,0.0263,729
273,"Tyler, The Creator",16,444,25.26,1,88.39,0.0255,705
227,SZA,3,400,26.24,3,91.67,0.0233,646


## 8. Popularity Trend Analysis

Popularity is analyzed over time using daily average popularity and a 7-day rolling average.

The rolling average helps identify sustained changes in playlist popularity while reducing short-term daily fluctuations.

In [17]:
daily_popularity = (
    df.groupby("date")
    .agg(
        average_popularity=("popularity", "mean"),
        median_popularity=("popularity", "median")
    )
    .reset_index()
    .sort_values("date")
)

daily_popularity["popularity_trend_7d"] = (
    daily_popularity["average_popularity"]
    .rolling(window=7, min_periods=1)
    .mean()
)

daily_popularity.head(10)

,date,average_popularity,median_popularity,popularity_trend_7d
0,2024-05-18,90.06,90.0,90.060000
1,2024-05-19,90.00,90.0,90.030000
2,2024-05-20,90.04,90.0,90.033333
3,2024-05-21,88.68,90.0,89.695000
4,2024-05-22,88.16,90.0,89.388000
5,2024-05-23,88.58,90.0,89.253333
6,2024-05-24,89.02,90.0,89.220000
7,2024-05-25,89.82,90.0,89.185714
8,2024-05-26,89.96,90.0,89.180000
9,2024-05-27,89.84,90.0,89.151429


In [18]:
print("Number of daily records:", len(daily_popularity))
print("Missing 7-day trend values:",
      daily_popularity["popularity_trend_7d"].isna().sum())

print("\nPopularity trend statistics:")
print(
    daily_popularity[
        ["average_popularity", "median_popularity", "popularity_trend_7d"]
    ].describe()
)

Number of daily records: 554
Missing 7-day trend values: 0

Popularity trend statistics:
       average_popularity  median_popularity  popularity_trend_7d
count          554.000000         554.000000           554.000000
mean            87.652383          88.741877            87.681959
std              4.910076           3.325497             2.468792
min             46.480000          46.500000            79.374286
25%             87.005000          88.000000            86.603571
50%             88.520000          89.000000            88.128571
75%             89.815000          90.500000            89.305714
max             92.280000          92.500000            91.717143


## 9. Popularity by Playlist Rank Range

Popularity is compared across the Top 10, Top 20, and Top 50 positions to examine how popularity varies across different levels of playlist ranking.

In [21]:
rank_group_popularity = pd.DataFrame({
    "rank_group": ["Top 10", "Top 20", "Top 50"],
    "average_popularity": [
        df.loc[df["position"] <= 10, "popularity"].mean(),
        df.loc[df["position"] <= 20, "popularity"].mean(),
        df.loc[df["position"] <= 50, "popularity"].mean()
    ],
    "median_popularity": [
        df.loc[df["position"] <= 10, "popularity"].median(),
        df.loc[df["position"] <= 20, "popularity"].median(),
        df.loc[df["position"] <= 50, "popularity"].median()
    ],
    "song_observations": [
        (df["position"] <= 10).sum(),
        (df["position"] <= 20).sum(),
        (df["position"] <= 50).sum()
    ]
})

rank_group_popularity["average_popularity"] = (
    rank_group_popularity["average_popularity"].round(2)
)

rank_group_popularity["median_popularity"] = (
    rank_group_popularity["median_popularity"].round(2)
)

rank_group_popularity

,rank_group,average_popularity,median_popularity,song_observations
0,Top 10,89.61,93.0,5540
1,Top 20,88.59,91.0,11080
2,Top 50,87.65,89.0,27700


## 10. Explicit Content Analysis

This analysis compares explicit and non-explicit tracks by playlist rank and popularity.

The goal is to understand how explicit content is distributed across the historical Top 50 playlist.

In [22]:
explicit_performance = (
    df.groupby("is_explicit")
      .agg(
          track_observations=("song", "count"),
          unique_songs=("song", "nunique"),
          average_rank=("position", "mean"),
          median_rank=("position", "median"),
          average_popularity=("popularity", "mean"),
          median_popularity=("popularity", "median")
      )
      .reset_index()
)

explicit_performance["average_rank"] = (
    explicit_performance["average_rank"].round(2)
)

explicit_performance["average_popularity"] = (
    explicit_performance["average_popularity"].round(2)
)

explicit_performance

,is_explicit,track_observations,unique_songs,average_rank,median_rank,average_popularity,median_popularity
0,False,14427,458,25.60,25.0,87.98,89.0
1,True,13273,499,25.39,26.0,87.30,89.0


In [23]:
explicit_by_rank = (
    df.groupby("rank_group", observed=True)
      .agg(
          explicit_share=("is_explicit", "mean"),
          average_popularity=("popularity", "mean"),
          track_observations=("song", "count")
      )
      .reset_index()
)

explicit_by_rank["explicit_share"] = (
    explicit_by_rank["explicit_share"] * 100
).round(2)

explicit_by_rank["average_popularity"] = (
    explicit_by_rank["average_popularity"].round(2)
)

explicit_by_rank

,rank_group,explicit_share,average_popularity,track_observations
0,Top 10,49.26,89.61,5540
1,Top 20,46.66,87.58,5540
2,Top 50,47.89,87.02,16620


## 11. Release Format Analysis

This analysis compares playlist performance across Single, Album, and Compilation tracks.

The comparison uses rank, popularity, duration, and playlist observations.

In [26]:
album_type_performance = (
    df.groupby("album_type")
      .agg(
          track_observations=("song", "count"),
          unique_songs=("song", "nunique"),
          average_rank=("position", "mean"),
          median_rank=("position", "median"),
          average_popularity=("popularity", "mean"),
          median_popularity=("popularity", "median"),
          average_duration_ms=("duration_ms", "mean")
      )
      .reset_index()
)

album_type_performance["average_rank"] = (
    album_type_performance["average_rank"].round(2)
)

album_type_performance["average_popularity"] = (
    album_type_performance["average_popularity"].round(2)
)

album_type_performance["average_duration_minutes"] = (
    album_type_performance["average_duration_ms"] / 60000
).round(2)

album_type_performance

,album_type,track_observations,unique_songs,average_rank,median_rank,average_popularity,median_popularity,average_duration_ms,average_duration_minutes
0,album,19672,745,26.10,26.0,86.67,88.0,201714.586163,3.36
1,compilation,42,8,33.19,36.0,78.38,82.0,200773.047619,3.35
2,single,7986,239,23.99,23.0,90.13,91.0,191127.795768,3.19


## 12. Track Duration Analysis

This analysis examines track duration in relation to playlist rank and popularity.

The goal is to understand whether track length is associated with playlist performance.

In [27]:
duration_summary = df["duration_ms"].describe().round(2)

print(duration_summary)

count     27700.00
mean     198660.95
std       48046.29
min       37314.00
25%      166453.00
50%      189386.00
75%      228013.00
max      740010.00
Name: duration_ms, dtype: float64


In [31]:
duration_analysis = (
    df.groupby("rank_group", observed=True)
    .agg(
average_duration_minutes=("duration_ms", lambda x: x.mean() / 60000),
median_duration_minutes=("duration_ms", lambda x: x.median() / 60000),
average_popularity=("popularity", "mean")
)
    .reset_index()
)

duration_analysis["average_duration_minutes"] = (
    duration_analysis["average_duration_minutes"].round(2)
)

duration_analysis["median_duration_minutes"] = (
    duration_analysis["median_duration_minutes"].round(2)
)

duration_analysis["average_popularity"] = (
    duration_analysis["average_popularity"].round(2)
)

duration_analysis

,rank_group,average_duration_minutes,median_duration_minutes,average_popularity
0,Top 10,3.32,3.11,89.61
1,Top 20,3.38,3.25,87.58
2,Top 50,3.28,3.13,87.02


## 13. Rank, Popularity and Duration Relationships

Correlation analysis is used to measure the strength and direction of relationships between playlist position, popularity, and track duration.

These results describe associations and should not be interpreted as causal effects.

In [30]:
correlation_matrix = (
    df[
        ["position", "popularity", "duration_ms"]
    ]
    .corr(method="spearman")
    .round(3)
)

correlation_matrix

,position,popularity,duration_ms
position,1.000,-0.245,-0.043
popularity,-0.245,1.000,-0.005
duration_ms,-0.043,-0.005,1.000


### Correlation Interpretation

The strongest observed relationship is between playlist position and popularity, with a Spearman correlation of -0.245.

Track duration shows very weak relationships with both playlist position (-0.043) and popularity (-0.005).

These correlations indicate association only and do not establish causation.